In [1]:
from dotenv import load_dotenv
load_dotenv()

from unsloth import is_bf16_supported, FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator

import json
import os
from omegaconf import OmegaConf
from PIL import Image
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import time
from trl import SFTTrainer, SFTConfig
from tqdm.notebook import tqdm
import wandb

from beyond_hate.train.utils import extract_label, convert_to_conversation_inference, resize_and_pad

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
cfg = OmegaConf.load("../../config/default.yaml")
config = cfg.training # Load the training configuration

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    config.model,
    load_in_4bit = config.load_in_4bit,
    use_gradient_checkpointing = config.use_gradient_checkpointing, 
    max_seq_length = config.max_seq_length
)

In [ ]:
# Load hate meme train set
hf_path = cfg.data.paths.hf
with open(f'{hf_path}/train.jsonl', 'r') as f:
    train_data = [json.loads(line) for line in f]
# Just keep items with valid image paths
train_data = [item for item in train_data if os.path.exists(f"{hf_path}/{item['img']}")]

In [3]:
SYSTEM_TEXT = ("You are a content moderation assistant. Aid me to"
                " label images with text as hateful or neutral."
                " Hateful image are defined as containing a direct or indirect"
                " attack on people based on characteristics, including"
                " ethnicity, race, nationality, immigration status, religion,"
                " caste, sex, gender identity, sexual orientation, and"
                " disability or disease.")

USER_TEXT = (' Considering the image and its text: "{}".'
             ' Is the content of the image and its text hateful or neutral? '
             ' Respond only with the word "Hateful" or "Neutral".')

In [ ]:
from beyond_hate.train.utils import HateMemeDataset

train_dataset = HateMemeDataset(train_data, SYSTEM_TEXT, USER_TEXT, hf_path,
                                size=tuple(config.img_size or []), color_padding=tuple(config.img_color_padding or []))

In [ ]:
# Configure LoRA to enable finetuning
model = FastVisionModel.get_peft_model(model,
                                       r=config.lora_r,
                                       lora_alpha=config.lora_alpha,
                                       max_seq_length=config.max_seq_length)

Unsloth: Making `base_model.model.model.vision_tower.vision_model` require gradients


In [ ]:
current_time = time.strftime("%y%m%d-%H%M")
output_dir = f'{cfg.out.runs}/{current_time}'
wandb.init(project=cfg.wandb.project, name=current_time, dir=cfg.out.path, config=dict(config))
    
FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_dataset,
    args = SFTConfig(
        per_device_train_batch_size = config.per_device_train_batch_size,
        gradient_accumulation_steps = config.gradient_accumulation_steps,
        warmup_steps = config.warmup_steps,
        num_train_epochs = config.num_train_epochs,
        learning_rate = config.learning_rate,
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        logging_steps = config.logging_steps,
        optim = config.optim,
        weight_decay = config.weight_decay,
        lr_scheduler_type = config.lr_scheduler_type,
        seed = config.seed,
        output_dir = output_dir,
        report_to = config.report_to,

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = config.remove_unused_columns,
        dataset_text_field = config.dataset_text_field,
        dataset_kwargs = {"skip_prepare_dataset": True},
        dataset_num_proc = config.dataset_num_proc,
        max_seq_length = config.max_seq_length,

        # Save strategy
        save_strategy = config.save_strategy,
        save_total_limit = config.save_total_limit,
    ),
)

trainer.train()

In [4]:
# Load hate meme dev set
hf_path = cfg.data.paths.hf
with open(f'{hf_path}/test_seen.jsonl', 'r') as f:
    val_data = [json.loads(line) for line in f]
# Just keep items with valid image paths
val_data = [item for item in val_data if os.path.exists(f"{hf_path}/{item['img']}")]

In [5]:
run_name = "250708-1815"
checkpoint = f"/workspace/disinfo/out/runs/{run_name}/checkpoint-844"

model, tokenizer = FastVisionModel.from_pretrained(
    checkpoint,
    load_in_4bit=config.load_in_4bit,
    use_gradient_checkpointing=config.use_gradient_checkpointing,
    max_seq_length=config.max_seq_length
)

==((====))==  Unsloth 2025.6.4: Fast Llava_Next patching. Transformers: 4.52.4.
   \\   /|    NVIDIA RTX 4000 Ada Generation. Num GPUs = 1. Max memory: 19.575 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.30G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [13]:
def unsloth_preprocessing(image, model):
    """
    Mimic the exact image preprocessing done by UnslothVisionDataCollator during training.
    This ensures consistency between training and inference.
    """
    # Get the image size used by the data collator
    try:
        image_size = model.config.vision_config.image_size
    except:
        print("Model does not have a default image size - using 512")
        image_size = 512
    
    # Apply the same resizing logic as UnslothVisionDataCollator
    if image_size is not None:
        if type(image_size) is tuple:
            # If tuple, resize to exact dimensions
            image = image.resize(image_size, Image.Resampling.LANCZOS)
        elif image.size[0] > image_size:
            # If single number and image width > size, resize maintaining aspect ratio
            wpercent = image_size / image.size[0]
            hsize = int(image.size[1] * wpercent)
            image = image.resize((image_size, hsize), Image.Resampling.LANCZOS)
    
    return image

In [6]:
# TODO: Batch processing
FastVisionModel.for_inference(model)

results = []
for sample in tqdm(val_data):
    label = sample['label']
    text = sample['text']
    
    # Resize and pad the image if specified in the config
    image = resize_and_pad(Image.open(f"{hf_path}/{sample['img']}"), target_size=tuple(config.img_size), color=(255, 255, 255))
    #image = unsloth_preprocessing(Image.open(f"{hf_path}/{sample['img']}"), model)
    #image = Image.open(f"{hf_path}/{sample['img']}")
    
    conversation = convert_to_conversation_inference(text, SYSTEM_TEXT, USER_TEXT)
    prompt = tokenizer.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = tokenizer(images=image, text=prompt, return_tensors="pt").to("cuda:0")

    # autoregressively complete prompt
    max_new_tokens = 50
    output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    output = tokenizer.decode(output[0], skip_special_tokens=True)
    output = output.split('[/INST]')[-1]

    results.append(
        {
            'id': sample['id'],
            'label': label,
            'output': output,
        }
    )

# Relabel
possible_labels = {'Hateful': 1, 'Neutral': 0}
y_true = [r['label'] for r in results]
#y_true = [1 if pred == 'Hateful' else 0 for pred in y_true]
y_pred = [extract_label(r['output'], possible_labels) for r in results]

# Filter out invalid predictions (-1)
valid = [i for i, pred in enumerate(y_pred) if pred != -1]
y_true_valid = [y_true[i] for i in valid]
y_pred_valid = [y_pred[i] for i in valid]
print(f'Invalid predictions: {len(y_pred) - len(y_true_valid)}')

accuracy = accuracy_score(y_true_valid, y_pred_valid)
precision = precision_score(y_true_valid, y_pred_valid, average='weighted')  # or 'macro' depending on your needs
recall = recall_score(y_true_valid, y_pred_valid, average='weighted')
f1 = f1_score(y_true_valid, y_pred_valid, average='weighted')
cm = confusion_matrix(y_true_valid, y_pred_valid, normalize='true')

  0%|          | 0/815 [00:00<?, ?it/s]

You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


Invalid predictions: 1


In [7]:
wandb.init(project=cfg.wandb.project, name = "250708-1815", id="esu3k5tt", resume="allow")

wandb.log({
    "test/accuracy": accuracy,
    "test/f1_score": f1,
    "test/invalid_prediction_rate": (len(y_pred) - len(y_true_valid)) / len(y_pred)
    })

wandb.log({"test/confusion_matrix": wandb.plot.confusion_matrix(
    probs=None,
    y_true=y_true_valid,
    preds=y_pred_valid,
    class_names=['Neutral', 'Hateful']
)})

wandb.finish()

wandb: Currently logged in as: nils_herrmann (nils_herrmann-technical-university-of-munich) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


test/accuracy,▁
test/f1_score,▁
test/invalid_prediction_rate,▁
eval/accuracy,0.74121
eval/f1_score,0.73635
eval/invalid_prediction_rate,0
eval/precision,0.7593
eval/recall,0.74121
eval/total_samples,398
test/accuracy,0.73833
test/f1_score,0.73792


Experiment 1:
- Model trained on resize and pad images 512x512
- Accuracy on resize and pad images 512x512: 0.6985
- Accuracy on resize and pad images 336x336: 0.6985
- Accuracy on unsloth resize: 0.6859
- Accuracy on raw: 0.6030

Experiment 2:
- Model trained on raw images
- Accuracy on resize and pad images 336x336: 0.6658
- Accuracy on unsloth resize: 0.5854
- Accuracy on raw: 

In [ ]:
# Log metrics to wandb
wandb.log({
    "eval/accuracy": accuracy,
    "eval/precision": precision,
    "eval/recall": recall,
    "eval/f1_score": f1,
    "eval/total_samples": len(y_true),
    "eval/valid_predictions": len(y_true_valid),
    "eval/invalid_predictions": len(y_pred) - len(y_true_valid),
    "eval/invalid_prediction_rate": (len(y_pred) - len(y_true_valid)) / len(y_pred)
})

wandb.log({"eval/confusion_matrix": wandb.plot.confusion_matrix(
    probs=None,
    y_true=y_true_valid,
    preds=y_pred_valid,
    class_names=['Neutral', 'Hateful']
)})

# Finish wandb run
wandb.finish()